<a href="https://colab.research.google.com/github/GarvGupta25/seq2seq-vs-attention-translation/blob/main/rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
!pip install -q torchtext

In [3]:
!pip install -q datasets

from datasets import load_dataset

# Load the dataset
data = load_dataset("bentrevett/multi30k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.tensorboard import SummaryWriter

In [5]:
print('train'[0])

t


In [6]:
import spacy

In [7]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 120.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 115.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [8]:
en_nlp = spacy.load('en_core_web_sm')
de_nlp = spacy.load('de_core_news_sm')

In [9]:
def tokenize_en(text):
  return [token.text for token in en_nlp.tokenizer(text)]
def tokenize_de(text):
    return [token.text for token in de_nlp.tokenizer(text)]
sample_text = data['train'][0]['en']
print(f"Original: {sample_text}")
print(f"Tokenized: {tokenize_en(sample_text)}")

Original: Two young, White males are outside near many bushes.
Tokenized: ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [10]:
def tokenize_dataset(example):
    example['en'] = tokenize_en(example['en'])
    example['de'] = tokenize_de(example['de'])
    return example
tokenized_dataset = data.map(tokenize_dataset)
print("Old format:", data['train'][0]['en'])
print("New format:", tokenized_dataset['train'][0]['en'])

Old format: Two young, White males are outside near many bushes.
New format: ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [11]:
from collections import Counter
en_counts = Counter()
de_counts = Counter()
for example in tokenized_dataset['train']:
    en_counts.update(example['en'])
    de_counts.update(example['de'])
en_vocab = {word: i+4 for i, (word, count) in enumerate(en_counts.items()) if count > 1}
de_vocab = {word: i+4 for i, (word, count) in enumerate(de_counts.items()) if count > 1}
special_tokens = {"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
en_vocab.update(special_tokens)
de_vocab.update(special_tokens)
print(f"Value of 'young': {en_vocab.get('young', 'Not Found')}")


Value of 'young': 5


In [12]:
def numericalize(example):
    # For every word in the English list, look up its number in en_vocab
    # If the word isn't in en_vocab (a typo or rare word), use the <unk> (0) token
    example['en'] = [en_vocab.get(word, 0) for word in example['en']]
    example['de'] = [de_vocab.get(word, 0) for word in example['de']]
    return example

# Apply this to your tokenized dataset
numericalized_dataset = tokenized_dataset.map(numericalize)

In [13]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
def collate_batch(batch):
    src_list = []
    trg_list = []
    for item in batch:
        # We manually inject <sos> (2) at the start and <eos> (3) at the end of every sentence
        src_tensor = torch.tensor([2] + item['en'] + [3])
        trg_tensor = torch.tensor([2] + item['de'] + [3])

        src_list.append(src_tensor)
        trg_list.append(trg_tensor)

    # By removing batch_first=True, PyTorch automatically stacks them as [seq_len, batch_size]
    # which is exactly what our LSTM needs!
    src_padded = pad_sequence(src_list, padding_value=1)
    trg_padded = pad_sequence(trg_list, padding_value=1)

    return src_padded, trg_padded

# Re-initialize your loader with the fixed collate function
train_loader = DataLoader(numericalized_dataset['train'], batch_size=64, collate_fn=collate_batch)

In [14]:
import torch
import torch.nn as nn
import random

class Encoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
    super(Encoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.dropout = nn.Dropout(p)
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)

  def forward(self, x):
    embedding = self.dropout(self.embedding(x))
    outputs, (hidden, cell) = self.rnn(embedding)
    return hidden, cell

class Decoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, output_size, num_layers, p):
    super(Decoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.dropout = nn.Dropout(p)
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)
    self.fc = nn.Linear(hidden_size, output_size)

  # FIXED: Moved def forward to the left so it is on the same level as def __init__
  def forward(self, x, hidden, cell):
    # FIXED: Indented all of these lines so they sit INSIDE the forward function
    x = x.unsqueeze(0)
    embedding = self.dropout(self.embedding(x))
    outputs, (hidden, cell) = self.rnn(embedding, (hidden, cell))
    # Removed the accidental duplicate self.rnn line here
    predictions = self.fc(outputs)
    predictions = predictions.squeeze(0)
    return predictions, hidden, cell

class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super(Seq2Seq, self).__init__()
    self.encoder = encoder
    self.decoder = decoder

  # FIXED: Added the missing colon at the end of this line
  def forward(self, source, target, teacher_force_ratio=0.5):
    batch_size = source.shape[1]
    target_len = target.shape[0]

    # FIXED: Using self.decoder.fc.out_features to dynamically get vocab size
    target_vocab_size = self.decoder.fc.out_features
    outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(source.device)

    hidden, cell = self.encoder(source)
    x = target[0]

    for t in range(1, target_len):
      output, hidden, cell = self.decoder(x, hidden, cell)
      outputs[t] = output
      best_guess = output.argmax(1)
      x = target[t] if random.random() < teacher_force_ratio else best_guess

    return outputs

In [15]:
# Correct vocab sizes
input_size_en = max(en_vocab.values()) + 1
input_size_de = max(de_vocab.values()) + 1

print(input_size_en, input_size_de)

# Then immediately after:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_size = 256
hidden_size = 512
num_layers = 2
dropout = 0.5
learning_rate = 0.001
num_epochs = 10
step = 0

encoder = Encoder(input_size_en, embedding_size, hidden_size, num_layers, dropout)
decoder = Decoder(input_size_de, embedding_size, hidden_size, input_size_de, num_layers, dropout)
model = Seq2Seq(encoder, decoder).to(device)

writer = SummaryWriter()

10628 19033


In [16]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
pad_idx = 1
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

for epoch in range(num_epochs):
    print(f"--- Starting Epoch {epoch+1} / {num_epochs} ---")
    model.train()

    # FIXED: Changed epoch.loss to epoch_loss
    epoch_loss = 0

    for batch_idx, (src_data, trg_data) in enumerate(train_loader):
        src_data = src_data.to(device)
        trg_data = trg_data.to(device)

        output = model(src_data, trg_data)
        output = output[1:].view(-1, output.shape[-1])
        trg_data = trg_data[1:].view(-1)

        optimizer.zero_grad()
        loss = criterion(output, trg_data)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()
        writer.add_scalar("Training Loss", loss.item(), global_step=step)
        step += 1

    # FIXED: Shifted these two lines to the left so they only run ONCE per epoch
    avg_loss = epoch_loss / len(train_loader)

    # FIXED: Removed the stray '/' at the very end of this line
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

--- Starting Epoch 1 / 10 ---
Epoch 1 completed. Average Loss: 5.0202
--- Starting Epoch 2 / 10 ---
Epoch 2 completed. Average Loss: 4.3280
--- Starting Epoch 3 / 10 ---
Epoch 3 completed. Average Loss: 3.9898
--- Starting Epoch 4 / 10 ---
Epoch 4 completed. Average Loss: 3.7781
--- Starting Epoch 5 / 10 ---
Epoch 5 completed. Average Loss: 3.6287
--- Starting Epoch 6 / 10 ---
Epoch 6 completed. Average Loss: 3.4860
--- Starting Epoch 7 / 10 ---
Epoch 7 completed. Average Loss: 3.3655
--- Starting Epoch 8 / 10 ---
Epoch 8 completed. Average Loss: 3.2406
--- Starting Epoch 9 / 10 ---
Epoch 9 completed. Average Loss: 3.1562
--- Starting Epoch 10 / 10 ---
Epoch 10 completed. Average Loss: 3.0552


In [17]:
def translate(sentence, en_vocab, de_vocab, model, device, max_len=50):
    model.eval()
    tokens = [en_vocab.get(word, 0) for word in tokenize_en(sentence)]
    src = torch.tensor([2] + tokens + [3]).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src)

    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    x = torch.tensor([2]).to(device)
    translated = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(x, hidden, cell)
        pred = output.argmax(1)
        if pred.item() == 3:  # <eos>
            break
        translated.append(de_idx_to_word.get(pred.item(), '<unk>'))
        x = pred

    return " ".join(translated)

# Test it
print(translate("A dog is running in the park", en_vocab, de_vocab, model, device))

Hund läuft im Wasser .


In [22]:
def evaluate_bleu(dataset, en_vocab, de_vocab, model, device, n=500):
    references = []
    hypotheses = []
    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    en_idx_to_word = {v: k for k, v in en_vocab.items()}

    test_data = dataset['test']

    for i in range(min(n, len(test_data))):
        example = test_data[i]

        # Convert indices back to words to feed into translate()
        src_words = [en_idx_to_word.get(idx, '<unk>')
                     for idx in example['en']
                     if idx not in [0, 1, 2, 3]]
        src_sentence = " ".join(src_words)

        # Get model translation
        translated = translate(src_sentence, en_vocab, de_vocab, model, device)

        # Get reference German words
        ref = [de_idx_to_word.get(idx, '')
               for idx in example['de']
               if idx not in [0, 1, 2, 3]]
        hyp = translated.split()

        references.append([ref])
        hypotheses.append(hyp)

    score = corpus_bleu(references, hypotheses)
    print(f"BLEU Score: {score*100:.2f}")
    return score

evaluate_bleu(numericalized_dataset, en_vocab, de_vocab, model, device)

BLEU Score: 12.82


0.12821321491813237